# Import 

In [ ]:
# WEEK 7 - CSGO DATASET:
# Xây dựng mô hình regression với bộ cs go
# Bỏ 2 cột: team_a_rounds và team_b_rounds. Đối với bài toán Regression, target là cột "points"

# Import libraries:
import pandas as pd
!pip install lazypredict
import lazypredict
from lazypredict.Supervised import LazyRegressor
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import OrthogonalMatchingPursuitCV, LassoLarsCV
from sklearn.model_selection import GridSearchCV

# Import dataset:
csgo_df = pd.read_csv(r'/Users/ngocta/Desktop/CoderSchool-AI/Week6_7/csgo', sep = ',')
target = 'points'

# View data statistics:
#csgo_profile = ProfileReport(csgo_df, title = 'CSGO Report', explorative = True)
#csgo_profile.to_file('CSGO Report.html')

# Transform datetime column:
csgo_df['date'] = pd.to_datetime(csgo_df['date'], dayfirst = True)

# Split to train & test sets:
x = csgo_df.drop(columns = ['team_a_rounds', 'team_b_rounds', target], axis = 1)
y = csgo_df[target]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 100)

# Handle numerical data:
num_cols = ['wait_time_s', 'match_time_s', 'ping', 'kills', 
            'assists', 'deaths', 'mvps', 'hs_percent']

num_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = "mean")),
    ('scaler', StandardScaler())
])

# Check the output:
#output = num_transformer.fit_transform(csgo_df[num_cols])
#for i, j in zip(csgo_df[num_cols].values, output):
    #print("Before: {}. After: {}".format(i,j))

#print(csgo_df.dtypes)
#print(x_test, y_test)

# Transform boolean text columns:
preprocessor = ColumnTransformer(transformers = [
    ('Numerical features', num_transformer, num_cols)
])

# Find the optimal model:
x_train_transformed = preprocessor.fit_transform(x_train)
x_test_transformed = preprocessor.transform(x_test)

clf = LazyRegressor(verbose = 0, ignore_warnings = True, custom_metric = None)
models, predictions = clf.fit(x_train_transformed, x_test_transformed, y_train, y_test)
print(predictions)

# Conclusion: Best models are OrthogonalMatchingPursuitCV & LassoLarsCV

In [ ]:
# Testing with 2 most optimal models:
def regression_report(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False)
    }
    
# Test with OrthogonalMatchingPursuitCV model
omp_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', OrthogonalMatchingPursuitCV(cv=5))
])
omp_pipeline.fit(x_train, y_train)
omp_predict = omp_pipeline.predict(x_test)
print("OrthogonalMatchingPursuitCV Results:")
print(regression_report(y_test, omp_predict))

# Test with LassoLarsCV model
lasso_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LassoLarsCV(cv=5))
])
lasso_pipeline.fit(x_train, y_train)
lasso_predict = lasso_pipeline.predict(x_test)
print("\n LassoLarsCV Results:")
print(regression_report(y_test, lasso_predict))
# Conclusion: Best regression model is OrthogonalMatchingPursuitCV

In [ ]:
# Finetuning OrthogonalMatchingPursuitCV:
from sklearn.linear_model import OrthogonalMatchingPursuit
from sklearn.model_selection import GridSearchCV

# Create pipeline with OrthogonalMatchingPursuitCV:
model_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', OrthogonalMatchingPursuit())
])

params = {
    'model__n_nonzero_coefs': [5, 10, 15, 20, None],
    'model__fit_intercept': [True, False]
}

# Finetune OrthogonalMatchingPursuitCV model with GridSearchCV:
model = GridSearchCV(
    estimator = model_pipeline,
    param_grid = params,
    scoring = 'neg_mean_squared_error',
    cv = 5,
    verbose = 1
)

# Predict using the finetuned model:
model.fit(x_train, y_train)
best_model = model.best_estimator_
finetuned_model = best_model.predict(x_test)

print("Best Parameters:", model.best_params_)
print("\n OrthogonalMatchingPursuit (GridSearchCV) Results:")
print(regression_report(y_test, omp_predict))